# Lesson 09: How large language models are built

*Worked solutions for [the lesson page](../../lessons/09-how-llms-are-built.html). The exercise version without solutions is `../09-how-llms-are-built.ipynb`.*

The full life of an LLM, from tokenizer to pretraining at scale, then instruction tuning, RLHF, DPO and reasoning training, ending with how decoding settings shape what the model says and why it sometimes makes things up.

**Runs on:** Any computer (CPU).

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy tiktoken

## Tokenization

### Byte-pair encoding, implemented

BPE starts from the 256 possible byte values and repeatedly merges the most frequent adjacent pair into a new token. Real tokenizers first split text into word-like chunks with a regular expression, so merges never cross word boundaries.

In [ ]:
import re
from collections import Counter

PAT = re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")   # pre-split into word-like chunks

def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

def train_bpe(text, vocab_size):
    chunks = [list(c.encode("utf-8")) for c in PAT.findall(text)]
    merges = {}                                  # (id_a, id_b) -> new_id, in the order learned
    for new_id in range(256, vocab_size):
        counts = Counter()
        for ch in chunks:
            counts.update(zip(ch, ch[1:]))
        if not counts:
            break
        pair = max(counts, key=counts.get)       # most frequent adjacent pair
        chunks = [merge(ch, pair, new_id) for ch in chunks]
        merges[pair] = new_id
    return merges

def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), new_id in merges.items():
        vocab[new_id] = vocab[a] + vocab[b]
    return vocab

def encode(text, merges):
    out = []
    for chunk in PAT.findall(text):
        ids = list(chunk.encode("utf-8"))
        while len(ids) >= 2:
            # apply the earliest-learned merge available, exactly as in training
            pair = min(set(zip(ids, ids[1:])), key=lambda p: merges.get(p, float("inf")))
            if pair not in merges:
                break
            ids = merge(ids, pair, merges[pair])
        out.extend(ids)
    return out

def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

In [ ]:
text = """Language models read text as tokens. A tokenizer splits text into pieces that are
common enough to deserve their own id. Frequent words become single tokens, rare words are
split into several pieces, and anything at all can fall back to raw bytes. The tokenizer is
trained before the model, on a large sample of the training data. Training finds the most
frequent pair of adjacent tokens, merges it into a new token, and repeats until the vocabulary
is large enough. Models, modelling, modelled and remodel share the piece "model", which helps
the model generalise across related words. Numbers like 2026 and 3.14159 are tokenized too,
often in awkward ways, which is one reason models make arithmetic mistakes.""" * 3

merges = train_bpe(text, 400)
vocab = build_vocab(merges)
print("first merges:", [vocab[i] for i in list(merges.values())[:10]])
for s in ["The tokenizer splits words.", "remodelling", "Tokenization of 3.14159", "naïve 🚀"]:
    ids = encode(s, merges)
    print(f"{s!r:<28} {len(s.encode()):>3} bytes -> {len(ids):>2} tokens:",
          [vocab[i].decode("utf-8", "replace") for i in ids])
    assert decode(ids, vocab) == s

Train it on a paragraph about tokenizers (repeated a few times) with a 400-token vocabulary, then look at what it does:

```text
'The tokenizer splits words.'   27 bytes ->  6 tokens: 'T' 'he' ' tokenizer' ' splits' ' words' '.'
'remodelling'                   11 bytes ->  4 tokens: 're' 'model' 'l' 'ing'
'Tokenization of 3.14159'       23 bytes -> 18 tokens: 'T' 'o' 'k' 'en' 'i' 'z' 'at' 'i' 'on' ' of' ...
'naïve 🚀'                      11 bytes -> 11 tokens: raw bytes for ï and the emoji
```

> **Tokens are the unit of everything**
>
> APIs charge per token, context windows are measured in tokens, and latency scales with tokens generated. Use the provider's token counter (or `tiktoken` for OpenAI-style tokenizers, or a Hugging Face tokenizer for open models) when you estimate costs. Never estimate from word counts in production.

## Decoding: turning distributions into text

- **Greedy**: always take the most likely token. Deterministic, but prone to dull, repetitive text.
- **Temperature**: divide logits by $T$ before softmax (you explored this in lesson 02). Below 1 sharpens, above 1 flattens.
- **Top-k**: sample only among the $k$ most likely tokens.
- **Top-p (nucleus)**: sample from the smallest set of tokens whose probabilities add up to $p$, so the candidate set adapts to how confident the model is.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

def sample(logits, temperature=1.0, top_k=None, top_p=None):
    p = softmax(logits / max(temperature, 1e-6))
    if top_k is not None:
        p = np.where(p >= np.sort(p)[-top_k], p, 0)
    if top_p is not None:
        order = np.argsort(-p)
        keep = order[: np.searchsorted(np.cumsum(p[order]), top_p) + 1]
        p = np.where(np.isin(np.arange(len(p)), keep), p, 0)
    p = p / p.sum()
    return rng.choice(len(p), p=p), p.round(3)

logits = np.array([3.0, 2.5, 1.0, 0.2, -1.0])
print(sample(logits))                    # full distribution [0.549 0.333 0.074 0.033 0.01 ]
print(sample(logits, top_k=2))           # [0.622 0.378 0.    0.    0.   ]
print(sample(logits, top_p=0.9))         # [0.574 0.348 0.078 0.    0.   ]
print(sample(logits, temperature=0.1))   # [0.993 0.007 0.    0.    0.   ]  nearly greedy

## The model landscape

### Hands-on: tokenizers in the wild

Install `tiktoken` (`uv add tiktoken`) and load `tiktoken.get_encoding("cl100k_base")`, a real production tokenizer.

1. Tokenize an English sentence, the same sentence translated into another language you know, a line of Python, and a long number. Compare token counts per character.
2. Print the individual tokens for "strawberry", " strawberry" and "Strawberry". Why do these differ, and how does this relate to letter-counting failures?
3. Remove the pre-split regex from the BPE code above (treat the whole text as one chunk) and retrain. Look at the longest learned tokens. What goes wrong?

#### Solution: what you should find

In [ ]:
# Downloads a model or data on first run
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
for s in ["The instrument was calibrated yesterday.",
          "Das Instrument wurde gestern kalibriert.",
          "for i in range(10): print(i ** 2)",
          "31415926535897932384"]:
    ids = enc.encode(s)
    print(len(s), len(ids), [enc.decode([t]) for t in ids])
for s in ["strawberry", " strawberry", "Strawberry"]:
    print(repr(s), [enc.decode([t]) for t in enc.encode(s)])

1. English is the most token-efficient; other languages usually need more tokens per character, so they cost more and fit less into the context window. Long numbers are split into chunks of a few digits.
2. A leading space and capitalisation produce different token sequences: " strawberry" and "strawberry" are distinct tokens or splits. The model sees a few opaque ids, not ten letters, so counting the r's requires knowledge it never directly observes.
3. Without pre-splitting, frequent multi-word phrases (including the trailing space and the next word) merge into single tokens. The vocabulary fills with phrase fragments that generalise poorly, and the same word gets different tokens depending on its neighbours.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/09-how-llms-are-built.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.